# Analyse TLS Handshake Results



# 1. Definitions

Run this section first to define the functions needed to analyse specific 
scenarios.

## 1.0 Globals

In [13]:
RESULTS_DIR = "./server-tls-results/"

## 1.1 Imports

In [71]:
import pandas as pd
import os
import scipy
import scipy.stats

# 1.2 Load a test set

In [ ]:
def load_algorithm_results(
    alg_results_dir: str,
    ind_var_shortname: str,
) -> pd.DataFrame:
    """Loads the results from a specific algorithm's results dir for a given test
    set.

    Arguments:
    - alg_results_dir: the directory containing that alg's results for a test 
    set.
    - alg_name: The long name of an algorithm for graphs etc.
    - ind_var_shorname: the shortname (e.g. CSV subheading) of the test's
    dependent variable.
    - ind_var_longname: The "long name" of the dependent variable e.g. "Packet 
    Loss Rate"
    """

    # load the file
    with open(alg_results_dir + "/batch-parameters.csv", "r") as csv:
        batch_parameters = pd.read_csv(csv)

    # read the parameters (e.g. "what was the packet loss rate for this batch?"
    # into a series object.
    batch_ind_var_values = pd.Series(
        data=batch_parameters[ind_var_shortname],
        index=batch_parameters["batch_number"]
    )
    #print(batch_ind_var_values)

    # load all the results into a DataFrame

    results = {} # it's easier to load them into a dict then convert that
    # dict to a DataFrame at the end, DataFrame's aren't built to be modfied
    # in-place very much.

    # read all of the batch-n.data files into one dataframe.
    for batch_number in batch_ind_var_values.index.to_list():
        batch_filename = f"{alg_results_dir}/batch-{batch_number}.data"
        ind_var = batch_ind_var_values[batch_number]
        with open(batch_filename, "r") as batch_file:
            batch_durations = batch_file.read().split(",")
            results[ind_var] = [float(duration) for duration in batch_durations]

    results_df = pd.DataFrame(results).replace(-1,None)
    return results_df


# 1.2 Statistically Analyse a test set

In [ ]:
def analyse_test_set(test_set: pd.DataFrame):
    # get the list of independent variable values
    batches = list(test_set)
    test_set_stats = {}

    # calculate useful statistics for each batch
    for batch_ind_var in batches:
        batch = test_set[batch_ind_var]
        test_set_stats["mean"] = batch.mean()
        # basic (i.e. non-corrected) bootstrap against the batch results.
        # get low and high CI value
        test_set_stats["CI_low"], test_set_stats["CI_high"] = scipy.stats.bootstrap(
            batch,
            method="basic").confidence_interval
        # calculate trimmed mean, i.e. mean of the middle 95%
        test_set_stats["trimmed_mean"] = scipy.stats.trim_mean(
            batch, 0.025
        )

    scipy.stats.bootstrap()

# X Testing

In [69]:
hqc_test = load_algorithm_results(f"{RESULTS_DIR}/latency/secLevel1/mlkem512/", "delay")
hqc_test[2.0].mean()

np.float64(2.919713677142857)